<a href="https://colab.research.google.com/github/rolandopmantilla/clases_py/blob/master/AHP_SDS_vs_APCH.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [62]:
import numpy as np

def calcular_prioridades(matriz):
    """Calcula el vector de prioridades (pesos) de una matriz de Saaty."""
    # Normalización: dividir cada elemento por la suma de su columna
    sumas_columnas = matriz.sum(axis=0)
    matriz_norm = matriz / sumas_columnas
    # El peso es el promedio de cada fila
    return matriz_norm.mean(axis=1)

In [63]:
# --- 1. DEFINIR IMPORTANCIA DE LOS CRITERIOS ---
# Matriz: [Educación, Social, Instalaciones]
# 1: Igual, 3: Moderado, 5: Fuerte, 1/3: Menos importante, 1/5 Mucho Menos Importante
matriz_criterios = np.array([
    [1,       3,    3],   # Educación frente a los otros
    [1/3,     1,    3],   # Social frente a los otros
    [1/3,   1/3,    1]    # Instalaciones frente a los otros
])

pesos_criterios = calcular_prioridades(matriz_criterios)

In [64]:
def calcular_consistencia(matriz, pesos):
    """Calcula el índice de consistencia (CI) y la razón de consistencia (CR) de una matriz de Saaty."""
    n = matriz.shape[0]

    # Paso 1: Calcular el vector suma ponderada (WSV)
    wsv = np.dot(matriz, pesos)

    # Paso 2: Calcular el lambda_max (promedio de la relación WSV/pesos)
    lambda_max = np.mean(wsv / pesos)

    # Paso 3: Calcular el Índice de Consistencia (CI)
    ci = (lambda_max - n) / (n - 1)

    # Paso 4: Obtener el Índice Aleatorio (RI) para n
    # Valores de RI para n=1 a n=10 (Fuente: Saaty)
    ri_valores = {1: 0, 2: 0, 3: 0.58, 4: 0.90, 5: 1.12, 6: 1.24, 7: 1.32, 8: 1.41, 9: 1.45, 10: 1.49}
    ri = ri_valores.get(n, 0) # Si n > 10, o no está en la tabla, se asume 0 para evitar errores, aunque debería ser manejado

    # Paso 5: Calcular la Razón de Consistencia (CR)
    cr = ci / ri if ri != 0 else np.inf # Evitar división por cero si n=1 o n=2 (o si ri_valores no tiene el valor)

    return ci, cr

# Calcular y mostrar la consistencia de la matriz de criterios
ci_criterios, cr_criterios = calcular_consistencia(matriz_criterios, pesos_criterios)
print(f"\nÍndice de Consistencia (CI) de Criterios: {ci_criterios:.4f}")
print(f"Razón de Consistencia (CR) de Criterios: {cr_criterios:.4f}")

# Interpretación de la consistencia
if cr_criterios < 0.10:
    print("La matriz de criterios es razonablemente consistente.")
else:
    print("ADVERTENCIA: La matriz de criterios tiene una consistencia pobre (CR >= 0.10). Considera revisar las comparaciones.")


Índice de Consistencia (CI) de Criterios: 0.0686
Razón de Consistencia (CR) de Criterios: 0.1183
ADVERTENCIA: La matriz de criterios tiene una consistencia pobre (CR >= 0.10). Considera revisar las comparaciones.


In [65]:
# --- 2. COMPARAR COLEGIOS (SDS vs APCH) POR CADA CRITERIO ---
# Escala: Si SDS es mejor que APCH, usa valores > 1. Si es peor, usa fracciones (1/3).

# Comparación en Educación (SDS preferido sobre APCH con un 5)
comp_edu = np.array([[1, 5], [1/5, 1]])
# Comparación en Ambiente Social (APCH preferido con un 5)
comp_soc = np.array([[1, 1/5], [5, 1]])
# Comparación en Instalaciones (SDS preferido con un 3)
comp_ins = np.array([[1, 3], [1/3, 1]])

# Obtener pesos locales
w_edu = calcular_prioridades(comp_edu)
w_soc = calcular_prioridades(comp_soc)
w_ins = calcular_prioridades(comp_ins)




In [66]:
# --- 3. CÁLCULO FINAL ---
# Unimos los resultados de los colegios en una sola matriz
matriz_alternativas = np.column_stack([w_edu, w_soc, w_ins])

# Multiplicamos los resultados por el peso de cada criterio
puntajes_finales = np.dot(matriz_alternativas, pesos_criterios)



In [67]:
# --- RESULTADOS ---
print(f"Pesos de Criterios: {pesos_criterios}")
print(f"Puntaje Final SDS: {puntajes_finales[0]:.4f}")
print(f"Puntaje Final APCH: {puntajes_finales[1]:.4f}")

Pesos de Criterios: [0.57362637 0.28644689 0.13992674]
Puntaje Final SDS: 0.6307
Puntaje Final APCH: 0.3693
